# Week 9 — Day 3
# Interactive Streamlit Dashboard

## Melanoma Skin Lesion Classification — Benign vs Malignant

This notebook continues directly from **Week 9 Day 2**. It wraps the validated CNN serving contract in a focused Streamlit interface for non-technical users.

### What this notebook delivers

- A clean Streamlit dashboard with an image uploader.
- The exact Day 2 preprocessing contract: **OpenCV → BGR to RGB → resize 128×128 → float32 → divide by 255**.
- A prominent prediction, malignant probability, decision threshold, and supporting visualization.
- A model-artifact setup that works in Google Colab.
- A local Streamlit launch command and a Colab browser link.



## Learning objectives

By the end of this notebook, you will be able to:

1. Explain why Streamlit is suitable for people while FastAPI is primarily suitable for programs.
2. Map an image input to the appropriate Streamlit widget.
3. Load the serialized Day 2 CNN instead of retraining it.
4. Display a prediction clearly and support it with a probability bar and image preview.
5. Run and review a focused demo suitable for a live presentation.

##  Why Streamlit?

A FastAPI endpoint exposes a contract for software clients. A Streamlit app exposes a friendly interface for people. Streamlit re-runs this top-to-bottom Python script whenever a widget changes, so the workflow stays simple: read the image, preprocess it, run the model, and render the result.



In [1]:
!pip -q install streamlit tensorflow opencv-python-headless pillow matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 121.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import shutil

BASE_DIR = Path('/content')
ARTIFACT_DIR = BASE_DIR / 'deployment_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Expected model:', ARTIFACT_DIR / 'melanoma_cnn.keras')
print('Expected metadata:', ARTIFACT_DIR / 'model_metadata.json')
print('The next cell lets you upload the Day 2 deployment package.')


Expected model: /content/deployment_artifacts/melanoma_cnn.keras
Expected metadata: /content/deployment_artifacts/model_metadata.json
The next cell lets you upload the Day 2 deployment package.


##  Bring in the Day 2 deployment artifacts

Upload the `deployment_artifacts.zip` created in Day 1/Day 2. The zip  contain :

```text
deployment_artifacts/
├── melanoma_cnn.keras   
└── model_metadata.json  
```



In [3]:
from google.colab import files
import zipfile

print('Choose deployment_artifacts.zip. If your artifacts are already present, cancel/skip this cell.')
uploaded = files.upload()
for name, data in uploaded.items():
    uploaded_path = BASE_DIR / name
    uploaded_path.write_bytes(data)
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(uploaded_path, 'r') as z:
            z.extractall(BASE_DIR)
        print('Extracted:', name)
    else:
        shutil.copy2(uploaded_path, ARTIFACT_DIR / name)
        print('Copied:', name)

for source_name, target_name in [
    ('model.keras', 'melanoma_cnn.keras'),
    ('metadata.json', 'model_metadata.json'),
]:
    source_candidates = [ARTIFACT_DIR / source_name, BASE_DIR / source_name]
    for source in source_candidates:
        if source.exists() and not (ARTIFACT_DIR / target_name).exists():
            shutil.copy2(source, ARTIFACT_DIR / target_name)
            break

print('\nArtifact files found:')
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(' -', p.name, f'({p.stat().st_size:,} bytes)')


Choose deployment_artifacts.zip. If your artifacts are already present, cancel/skip this cell.


Saving deployment_artifacts (1).zip to deployment_artifacts (1).zip
Extracted: deployment_artifacts (1).zip

Artifact files found:


In [8]:
from pathlib import Path
import urllib.request
import json
import tensorflow as tf

BASE_DIR = Path("/content")
ARTIFACT_DIR = BASE_DIR / "deployment_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "melanoma_cnn.keras"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

GITHUB_BASE = (
    "https://raw.githubusercontent.com/"
    "LunaYahya/AI-Training/main/"
    "Week9/Day2/deployment_airtifacts"
 )

if not MODEL_PATH.exists():
    print("Downloading melanoma_cnn.keras ...")
    urllib.request.urlretrieve(
        f"{GITHUB_BASE}/melanoma_cnn.keras",
        MODEL_PATH
    )

if not METADATA_PATH.exists():
    print("Downloading model_metadata.json ...")
    urllib.request.urlretrieve(
        f"{GITHUB_BASE}/model_metadata.json",
        METADATA_PATH
    )

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file was not found: {MODEL_PATH}"
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file was not found: {METADATA_PATH}"
    )

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

input_config = metadata["input"]


if "image_size" in input_config:
    IMG_SIZE = tuple(input_config["image_size"])
else:
    IMG_SIZE = (
        int(input_config["height"]),
        int(input_config["width"]),
    )

CHANNELS = int(input_config["channels"])

CLASS_TO_LABEL = metadata["classes"]

LABEL_TO_CLASS = {
    int(value): key
    for key, value in CLASS_TO_LABEL.items()
}

SELECTED_THRESHOLD = float(
    metadata["decision_rule"]["selected_threshold"]
)

assert IMG_SIZE == (128, 128), (
    f"Expected image size (128, 128), but received {IMG_SIZE}"
)

assert CHANNELS == 3, (
    f"Expected 3 channels, but received {CHANNELS}"
)

print("Deployment metadata loaded successfully.")
print("Image size:", IMG_SIZE)
print("Channels:", CHANNELS)
print("Classes:", CLASS_TO_LABEL)
print("Label mapping:", LABEL_TO_CLASS)
print("Selected threshold:", SELECTED_THRESHOLD)

model = tf.keras.models.load_model(MODEL_PATH)

print("\nCNN model loaded successfully.")
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)
expected_shape = (None, IMG_SIZE[0], IMG_SIZE[1], CHANNELS)

if tuple(model.input_shape) != expected_shape:
    raise ValueError(
        f"Unexpected model input shape: {model.input_shape}. "
        f"Expected: {expected_shape}"
    )

print("PASS: Model input shape matches the Day 2 deployment contract.")



Deployment metadata loaded successfully.
Image size: (128, 128)
Channels: 3
Classes: {'Benign': 0, 'Malignant': 1}
Label mapping: {0: 'Benign', 1: 'Malignant'}
Selected threshold: 0.35

CNN model loaded successfully.
Model input shape: (None, 128, 128, 3)
Model output shape: (None, 1)
PASS: Model input shape matches the Day 2 deployment contract.


##  Build the Streamlit app

The app below follows the recommended demo pattern:

- clear title and one-sentence description;
- sensible image uploader and threshold control;
- a single primary action;
- a prominent result card;
- probability visualization and uploaded-image preview;
- a small technical details section for reproducibility.

In [10]:
APP_PATH = BASE_DIR / 'app.py'
APP_PATH.write_text('from pathlib import Path\nimport json\nimport io\n\nimport cv2\nimport numpy as np\nimport streamlit as st\nfrom PIL import Image\nimport tensorflow as tf\n\n# ---------- Page setup ----------\nst.set_page_config(\n    page_title="Lesion Insight | Melanoma Classifier",\n    page_icon="🩺",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n# ---------- Styling ----------\nst.markdown("""\n<style>\n    @import url(\'https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Playfair+Display:wght@600;700&display=swap\');\n    .stApp { background: #f7f8fc; color: #172033; }\n    .block-container { max-width: 1180px; padding-top: 2.2rem; padding-bottom: 3rem; }\n    h1, h2, h3 { font-family: \'Playfair Display\', Georgia, serif; color: #172033; }\n    p, label, div, span { font-family: \'DM Sans\', Arial, sans-serif; }\n    .hero { background: linear-gradient(135deg, #18233f 0%, #2a4772 100%); padding: 2.4rem 2.7rem; border-radius: 22px; color: white; margin-bottom: 1.5rem; box-shadow: 0 14px 30px rgba(33, 55, 95, .16); }\n    .hero h1 { color: white; font-size: 2.6rem; margin: 0 0 .35rem 0; }\n    .hero p { color: #dbe6f8; font-size: 1.05rem; margin: 0; }\n    .eyebrow { text-transform: uppercase; letter-spacing: .15em; font-size: .72rem; font-weight: 700; color: #9ec5ff; margin-bottom: .8rem; }\n    .result { padding: 1.2rem 1.4rem; border-radius: 16px; border: 1px solid #dce5f1; background: white; box-shadow: 0 8px 20px rgba(30, 50, 80, .06); }\n    .result-good { border-left: 6px solid #2c9b72; }\n    .result-alert { border-left: 6px solid #d65a5a; }\n    .result-label { color: #68758b; font-size: .8rem; text-transform: uppercase; letter-spacing: .12em; font-weight: 700; }\n    .result-value { font-size: 2.1rem; font-weight: 700; margin-top: .25rem; }\n    .muted { color: #68758b; font-size: .92rem; }\n    .disclaimer { background: #fff8e8; border: 1px solid #f0d99a; color: #614d1a; padding: .9rem 1rem; border-radius: 12px; font-size: .88rem; }\n    [data-testid="stFileUploader"] { background: white; border-radius: 14px; padding: .4rem; }\n    .small-note { color: #77839a; font-size: .8rem; }\n</style>\n""", unsafe_allow_html=True)\n\n# ---------- Model and preprocessing ----------\nARTIFACT_DIR = Path(\'/content/deployment_artifacts\')\nMODEL_PATH = ARTIFACT_DIR / \'melanoma_cnn.keras\'\nMETADATA_PATH = ARTIFACT_DIR / \'model_metadata.json\'\n\n@st.cache_resource\ndef load_assets():\n    metadata = json.loads(METADATA_PATH.read_text(encoding=\'utf-8\'))\n    model = tf.keras.models.load_model(MODEL_PATH)\n    image_size = tuple(metadata[\'input\'][\'image_size\'])\n    threshold = float(metadata[\'decision_rule\'][\'selected_threshold\'])\n    labels = {int(v): k for k, v in metadata[\'classes\'].items()}\n    return model, metadata, image_size, threshold, labels\n\ndef preprocess_image(image_bytes, image_size):\n    buffer = np.frombuffer(image_bytes, dtype=np.uint8)\n    image = cv2.imdecode(buffer, cv2.IMREAD_COLOR)\n    if image is None:\n        raise ValueError(\'The uploaded file is not a readable image.\')\n    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n    image = cv2.resize(image, image_size, interpolation=cv2.INTER_AREA)\n    image = image.astype(np.float32) / 255.0\n    return np.expand_dims(image, axis=0)\n\n# ---------- Header ----------\nst.markdown("""<div class="hero"><div class="eyebrow">Interactive model demo · Week 9 Day 3</div><h1>Lesion Insight</h1><p>A focused, human-friendly interface for exploring the melanoma classification model.</p></div>""", unsafe_allow_html=True)\n\nif not MODEL_PATH.exists() or not METADATA_PATH.exists():\n    st.error(\'Deployment artifacts are missing. Please place melanoma_cnn.keras and model_metadata.json in /content/deployment_artifacts.\')\n    st.stop()\n\ntry:\n    model, metadata, image_size, default_threshold, labels = load_assets()\nexcept Exception as exc:\n    st.error(f\'Could not load the model: {exc}\')\n    st.stop()\n\n# ---------- Sidebar ----------\nwith st.sidebar:\n    st.markdown(\'## About this demo\')\n    st.write(\'Upload a clear skin-lesion image to receive the CNN prediction and its malignant probability.\')\n    threshold = st.slider(\'Decision threshold\', 0.00, 1.00, float(default_threshold), 0.01, help=\'A prediction is labelled Malignant when the malignant probability is at least this value.\')\n    st.divider()\n    st.markdown(\'### Serving contract\')\n    st.caption(f\'Input size: {image_size[0]} × {image_size[1]} pixels\')\n    st.caption(\'Color format: RGB\')\n    st.caption(\'Scaling: float32 / 255\')\n    st.caption(f\'Default threshold: {default_threshold:.2f}\')\n\n# ---------- Main interaction ----------\nleft, right = st.columns([1.05, .95], gap=\'large\')\nwith left:\n    st.markdown(\'### 1 · Choose an image\')\n    uploaded = st.file_uploader(\'Upload a lesion image\', type=[\'jpg\', \'jpeg\', \'png\', \'webp\'], help=\'Supported formats: JPG, JPEG, PNG, WEBP\')\n    if uploaded is None:\n        st.info(\'Upload an image to activate the prediction panel.\')\n    else:\n        preview = Image.open(io.BytesIO(uploaded.getvalue()))\n        st.image(preview, caption=f\'Uploaded image · {uploaded.name}\', use_container_width=True)\n\nwith right:\n    st.markdown(\'### 2 · Review the result\')\n    predict_clicked = st.button(\'Run prediction\', type=\'primary\', use_container_width=True)\n    if predict_clicked and uploaded is not None:\n        with st.spinner(\'Analyzing the image…\'):\n            try:\n                x = preprocess_image(uploaded.getvalue(), image_size)\n                probability = float(np.asarray(model.predict(x, verbose=0)).squeeze())\n                predicted_label = int(probability >= threshold)\n                predicted_class = labels[predicted_label]\n            except Exception as exc:\n                st.error(f\'Prediction failed: {exc}\')\n                st.stop()\n\n        is_malignant = predicted_label == 1\n        card_class = \'result-alert\' if is_malignant else \'result-good\'\n        st.markdown(f"""<div class="result {card_class}"><div class="result-label">Model prediction</div><div class="result-value">{predicted_class}</div><div class="muted">Decision threshold: {threshold:.2f}</div></div>""", unsafe_allow_html=True)\n        st.write(\'\')\n        st.metric(\'Malignant probability\', f\'{probability:.1%}\')\n        st.progress(min(max(probability, 0.0), 1.0), text=f\'Malignant probability · {probability:.1%}\')\n\n        chart_data = {\'Benign\': max(0.0, 1.0 - probability), \'Malignant\': probability}\n        st.bar_chart(chart_data, horizontal=True, height=150, color=\'#5379ad\')\n        st.caption(\'The chart shows the model score before applying the selected threshold.\')\n    elif uploaded is not None:\n        st.info(\'Ready when you are. Click **Run prediction** to continue.\')\n\nst.markdown(\'\')\nst.markdown(\'<div class="disclaimer"><strong>Educational demonstration:</strong> this classifier is not a medical diagnosis and should not replace evaluation by a qualified healthcare professional.</div>\', unsafe_allow_html=True)\n', encoding='utf-8')
print('Created:', APP_PATH)
print('Size:', APP_PATH.stat().st_size, 'bytes')


Created: /content/app.py
Size: 6936 bytes


##  Run the dashboard in Google Colab

The next cell starts Streamlit on port 8501. In Colab, `serve_kernel_port_as_window` opens the app through the notebook's built-in proxy. In a local Jupyter environment, open `http://localhost:8501` instead.

In [28]:
import subprocess
import time
from google.colab import output

PORT = 8503

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false",
        "--browser.gatherUsageStats",
        "false",
    ],
    stdout=open("/tmp/streamlit_8503.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(8)

print(f"Streamlit is running on port {PORT}.")

try:
    output.serve_kernel_port_as_iframe(PORT)
except AttributeError:
    output.serve_kernel_port_as_window(PORT)


Streamlit is running on port 8503.


<IPython.core.display.Javascript object>

In [25]:
from google.colab import output
from IPython.display import display, HTML

PORT = 8503

url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

print("Dashboard URL:")
print(url)

display(
    HTML(
        f'''
        <a href="{url}" target="_blank"
           style="
           display:inline-block;
           padding:12px 20px;
           background:#2a4772;
           color:white;
           border-radius:8px;
           text-decoration:none;
           font-weight:bold;
           ">
           Open Lesion Insight Dashboard
        </a>
        '''
    )
)


Dashboard URL:
https://8503-gpu-t4-s-kkb-ass1c2-f27cxi2cn1kt-c.asia-southeast1-2.prod.colab.dev


# Final Day 3 checklist

- [X] Streamlit app created
- [X] Week 8 CNN loaded
- [X] 128×128 RGB preprocessing preserved
- [X] Pixel normalization `/255.0` preserved
- [X] Threshold `0.35` preserved
- [X] File uploader implemented
- [X] Uploaded image displayed
- [X] Prediction displayed prominently
- [X] Confidence displayed
- [X] Probability visualization added
- [X] No-image case handled
- [X] Invalid file type handled
- [X] App tested
- [X] requirements.txt created
- [X] README created


## Outcome

The validated Week 8 CNN is now presented through a clean, interactive interface for non-technical users.

## Summary

This notebook completes Day 3 by turning the Day 2 serving contract into a usable Streamlit dashboard. It uses an image uploader, a threshold slider, a clear result card, a probability visualization, and a preview of the uploaded image. The app is designed to be easy to run in Google Colab and easy to explain during a live presentation.